<a href="https://colab.research.google.com/github/pawankeshri/langchain-demo/blob/main/CrewAI_Masterclass_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building a Multi-Agent System with CrewAI

Welcome to this hands-on Colab notebook on building a **multi-agent AI workflow using CrewAI**.

You will build a small but realistic agent system made of three specialized agents:

1. a **Research Analyst** that gathers evidence,
2. a **Content Writer** that converts research into a structured article,
3. a **Social Media Editor** that turns the article into platform-specific social drafts.

The key idea to take away is that a multi-agent system is **not just multiple prompts**. It is a coordinated workflow with clearly scoped agent roles, different goals and behaviors per agent, task handoffs, optional tool usage, structured outputs, and a final orchestration layer that runs everything in sequence.

## What you will learn

By the end of this notebook, you should understand:

- what CrewAI is and when to use it,
- how to define agents with distinct roles and goals,
- how to attach tools to agents,
- how to chain tasks across agents,
- how to use structured outputs for reliability,
- how to run a crew end to end,
- and how to save and inspect the outputs.

## What this notebook builds

The workflow follows a simple three-step pipeline:

**Research → Article Draft → Social Post Drafts**

This keeps the focus on core multi-agent concepts without introducing too many moving parts at once.

## Prerequisites

Before running the notebook, you should be comfortable with:

- basic Python notebook usage,
- what an API key is,
- the high-level idea of LLMs,
- and the difference between prompting and tool usage.

No prior CrewAI experience is required.

## How to run this notebook

Run the notebook **top to bottom**, one cell at a time.

- Run the install cell first.
- If Colab asks for a runtime restart, restart the runtime.
- After restarting, rerun the notebook from the top.
- Do not skip the secrets/configuration cells.
- Run the final publishing cell only if you explicitly want to test webhook publishing.

## Required keys

This notebook requires:

- `OPENAI_API_KEY` — required
- `SERPER_API_KEY` — optional, only needed for live web research
- `SOCIAL_WEBHOOK_URL` — optional, only needed for manual publishing

## Expected outputs

When the notebook completes successfully, it will generate:

- research notes,
- a structured article draft,
- LinkedIn and X/Twitter draft posts,
- and saved markdown files inside the `output/` folder.

## Cell 1 — Install the libraries needed for CrewAI

This cell prepares the notebook environment by installing the core library:

- `crewai[tools]` — the core framework plus optional tool integrations such as live search, installed together as a single package

This is the environment setup step that every Colab workflow needs before any CrewAI code can run.

### Why this matters

LLM application notebooks are sensitive to package versions. Many runtime errors come not from the logic of the agents, but from outdated packages, incompatible dependencies, or kernels that were not restarted after installation. So this cell is not just "setup" — it is part of production hygiene.

### How to run this cell

1. Run this cell **once** at the beginning of the notebook.
2. Wait for the installation to complete fully.
3. If Colab displays a message asking you to **restart the runtime**, do that immediately.
4. After restarting, come back to the notebook and rerun all cells from the top.

### What should happen after running

You should see `✅ crewai <version> installed successfully` at the bottom of the output.

> **Note:** Colab pre-installs several Google packages (google-adk, bigframes, etc.) whose pinned dependency versions can clash with CrewAI's dependencies. If you see yellow or red `dependency conflict` warnings during install, **they are safe to ignore** — they affect those Google packages, not CrewAI. The sanity-check line at the end of this cell confirms that CrewAI itself loaded correctly.

In [ ]:
# Install CrewAI (ignore dependency-conflict warnings from pre-installed Colab packages).
%pip install -qU "crewai[tools]==1.12.2" 2>&1 | grep -v "^ERROR: pip" | grep -v "incompatible"

# Quick sanity check — if this prints a version number, the install succeeded.
try:
    import crewai as _c
    print(f"\n✅ crewai {_c.__version__} installed successfully")
except ImportError:
    print("\n⚠️  Install complete — please restart the runtime (Runtime → Restart runtime) and rerun all cells.")



✅ crewai 1.12.2 installed successfully


## Cell 2 — Import the libraries and verify the runtime

This cell confirms that the notebook is running with the expected Python and CrewAI environment. It does three things:

- imports the libraries that the rest of the notebook depends on,
- prints the Python version,
- prints the CrewAI package versions that were actually loaded into memory.

### Why this matters

In notebooks, installation success does **not automatically guarantee** import success. You may install the right package but still have an older version cached in memory, or be using a runtime that is not compatible with the framework. This cell catches those problems early.

### How to run this cell

1. Run this cell right after the install step.
2. If the imports fail, do **not** keep going.
3. Restart the runtime and run the notebook again from the first cell.

### What should happen after running

You should see the Python version, the CrewAI version, and the CrewAI Tools version. If the runtime version is outside the supported range, the cell will print a warning.

> **Key takeaway:** Good AI workflow development includes **environment verification** before business logic. That habit saves a lot of debugging time later.

In [ ]:
import os
import sys
from pathlib import Path

import crewai
import crewai_tools
from IPython.display import Markdown, display

print(f"Python: {sys.version.split()[0]}")
print(f"CrewAI: {crewai.__version__}")
print(f"CrewAI Tools: {crewai_tools.__version__}")

if not ((3, 10) <= sys.version_info[:2] < (3, 14)):
    print("Warning: CrewAI 1.12.2 targets Python >=3.10 and <3.14.")

Python: 3.12.13
CrewAI: 1.12.2
CrewAI Tools: 1.12.2


## Cell 3 — Load API keys and configure notebook-safe execution

This cell handles two things that every real LLM application needs:

1. **Secure secret loading** — getting access to the OpenAI API key, the optional Serper key for live search, and the optional webhook URL for publishing.
2. **Runtime behavior control** — disabling tracing and telemetry that can interrupt notebook execution.

### Why this matters

A production-style agent notebook should never hardcode secrets directly in the notebook body. Instead, secrets should come from environment variables, secure notebook secret stores, or a secure input prompt. This cell supports all three approaches, with automatic fallback.

Notice the distinction between a **required dependency** (the OpenAI key, without which the workflow cannot run) and an **optional capability** (the Serper key, which expands what the workflow can do when available). That pattern appears often in agent systems.

### How to run this cell

1. Run this cell after the import/version check cell.
2. If your keys are already stored in Colab secrets or environment variables, the cell will use them automatically.
3. If a required key is missing, the notebook will prompt you to enter it securely.
4. Enter `OPENAI_API_KEY` when asked.
5. Enter `SERPER_API_KEY` only if you want the research agent to use live web search.
6. Enter `SOCIAL_WEBHOOK_URL` only if you plan to test manual publishing later.

### What should happen after running

The cell will print whether each secret was loaded:

- `OPENAI_API_KEY` should be `True`
- `SERPER_API_KEY` may be `True` or `False`
- `SOCIAL_WEBHOOK_URL` may be `True` or `False`

In [ ]:
import getpass
from typing import Optional

# Disable prompts/noise that can interrupt notebook execution.
os.environ["CREWAI_TRACING_ENABLED"] = "false"
os.environ["CREWAI_DISABLE_TELEMETRY"] = "true"


def load_secret(name: str, required: bool = True) -> Optional[str]:
    """Load a secret from env vars, Colab userdata, or a secure prompt."""
    value = os.getenv(name)
    if value:
        return value

    # Colab-only fallback. This block is ignored safely in regular Jupyter.
    try:
        from google.colab import userdata  # type: ignore

        value = userdata.get(name)
        if value:
            os.environ[name] = value
            return value
    except Exception:
        pass

    if required:
        value = getpass.getpass(f"Enter {name}: ").strip()
        if not value:
            raise ValueError(f"{name} is required to run this notebook.")
        os.environ[name] = value
        return value

    return None


OPENAI_API_KEY = load_secret("OPENAI_API_KEY", required=True)
SERPER_API_KEY = load_secret("SERPER_API_KEY", required=False)
SOCIAL_WEBHOOK_URL = load_secret("SOCIAL_WEBHOOK_URL", required=False)

print("OPENAI_API_KEY loaded:", bool(OPENAI_API_KEY))
print("SERPER_API_KEY loaded:", bool(SERPER_API_KEY))
print("SOCIAL_WEBHOOK_URL loaded:", bool(SOCIAL_WEBHOOK_URL))

OPENAI_API_KEY loaded: True
SERPER_API_KEY loaded: False
SOCIAL_WEBHOOK_URL loaded: False


## Cell 4 — Configure the topic, model, and execution settings

This cell defines the main configurable parameters of the multi-agent system. It answers four questions:

- **Which model should the agents use?**
- **What topic should the crew work on?**
- **Should live web research be enabled?**
- **Where should outputs be saved?**

Think of this as the notebook's **control panel**.

### Why this matters

A useful design principle in agent workflows is to keep the **logic** separate from the **configuration**. Instead of scattering hardcoded values throughout the notebook, this cell centralizes the main knobs you are likely to change. That makes the notebook easier to reuse, adapt for different topics, and debug.

Notice that the same agent architecture can support many domains simply by changing the `TOPIC` value here — no changes needed in the agent or task definitions downstream.

### How to run this cell

1. Read the current values before running.
2. Update `TOPIC` if you want to explore a different example.
3. Leave `MODEL_NAME` as is unless you specifically want to switch models.
4. Keep `AUTO_PUBLISH = False` for now.
5. Run the cell.

### What should happen after running

The cell will print the active configuration: selected model, selected topic, whether web research is enabled, whether auto-publishing is enabled, and the output folder path.

In [ ]:
MODEL_NAME = os.getenv("OPENAI_MODEL_NAME", "openai/gpt-4o-mini")
TOPIC = "Agentic Analytics for Data Analysts"

ENABLE_WEB_RESEARCH = bool(SERPER_API_KEY)
AUTO_PUBLISH = False  # Keep this off by default for safety.

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Model: {MODEL_NAME}")
print(f"Topic: {TOPIC}")
print(f"Web research enabled: {ENABLE_WEB_RESEARCH}")
print(f"Auto-publish enabled: {AUTO_PUBLISH}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

Model: openai/gpt-4o-mini
Topic: Agentic Analytics for Data Analysts
Web research enabled: False
Auto-publish enabled: False
Output directory: /content/output


## Cell 5 — Define schemas, tools, and helper functions

This cell introduces several foundational building blocks used in real multi-agent systems:

- **Structured outputs** using Pydantic
- **Tool attachment** for live research
- An optional **publishing helper** for downstream automation

### Part 1: Why structured outputs matter

The notebook defines two output schemas: `ArticleDraft` and `SocialDrafts`. These schemas tell the model exactly what shape the output should have.

This is extremely important in applied agent systems because free-form text is often hard to validate, reuse in downstream steps, save cleanly, or send into another tool. Structured outputs make multi-step workflows much more reliable.

### Part 2: Why tools matter

The `build_research_tools()` function adds a search tool only when the required search key is available. This demonstrates a key design pattern: agents can be given tools **conditionally**, and the rest of the system can adapt to the available capabilities. That is more robust than assuming every environment has every external dependency.

### Part 3: Why helper functions matter

The `publish_to_webhook()` helper is included as an optional extension. The key lesson here is that **generation and action should often be separated** — first generate drafts, then review them, then publish manually if needed.

### How to run this cell

1. Run this cell after configuration is complete.
2. Do not change the Pydantic schemas unless you want to change the output format.
3. Do not worry if the research tool list ends up empty — the notebook can still run without live web search.

### What should happen after running

The cell will create the schemas, the tool builder, the webhook helper, and the final `research_tools` list. It will also print whether any research tools were attached.

> **Key takeaway:** This cell shows how to move from "a prompt that produces text" to "a system that produces machine-usable outputs and can interact with external capabilities."  That shift is central to building real agent workflows.

In [ ]:
import json
from textwrap import dedent

import requests
from pydantic import BaseModel, Field

from crewai import Agent, Crew, LLM, Process, Task
from crewai_tools import SerperDevTool


class ArticleDraft(BaseModel):
    title: str = Field(..., description="A concise article headline.")
    summary: str = Field(..., description="Two or three sentences summarizing the article.")
    article_markdown: str = Field(
        ...,
        description="The article body in markdown with headings and a conclusion. Do not wrap it in code fences.",
    )


class SocialDrafts(BaseModel):
    linkedin_post: str = Field(..., description="A LinkedIn-ready draft with 3-5 relevant hashtags.")
    x_post: str = Field(..., description="An X/Twitter draft no longer than 280 characters.")
    publishing_status: str = Field(
        ...,
        description="A status line confirming that nothing has been published automatically.",
    )


def build_research_tools():
    """Attach live web search only when the Serper key is available."""
    if ENABLE_WEB_RESEARCH and os.getenv("SERPER_API_KEY"):
        return [SerperDevTool(search_type="search", n_results=5)]
    return []


def publish_to_webhook(text: str, image_path: Optional[str] = None) -> str:
    """Publish reviewed content manually via a webhook."""
    webhook_url = os.getenv("SOCIAL_WEBHOOK_URL")
    if not webhook_url:
        raise ValueError(
            "SOCIAL_WEBHOOK_URL is not set. Add it only when you are ready to publish manually."
        )

    if image_path:
        image_file = Path(image_path)
        if not image_file.exists():
            raise FileNotFoundError(f"Image not found: {image_file}")

        with image_file.open("rb") as handle:
            response = requests.post(
                webhook_url,
                data={"text": text},
                files={"image": handle},
                timeout=60,
            )
            response.raise_for_status()
    else:
        response = requests.post(webhook_url, data={"text": text}, timeout=60)
        response.raise_for_status()

    return f"Webhook accepted the post with status code {response.status_code}."


research_tools = build_research_tools()
print("Research tools attached:", [tool.name for tool in research_tools] or "None")

Research tools attached: None


## Cell 6 — Create the agents

This is the first true multi-agent design cell. Here we define the three agents that will collaborate in the workflow:

- **Research Analyst**
- **Economic Content Writer**
- **Social Media Editor**

Each agent has its own role, goal, backstory, tool access, reasoning style, and model configuration.

### Why this matters

A multi-agent system works best when each agent has a **clear responsibility boundary**. Instead of asking one general-purpose agent to do everything, we split the workflow into specialized roles. This helps with clarity, modularity, easier debugging, better task handoffs, and more predictable outputs.

Notice two important design choices in the code below:

1. The researcher may get tools, but the writer and social editor do not.
2. The writer and social editor use a slightly more creative temperature (`0.6`), while the researcher stays more conservative (`0.2`).

That is a practical example of assigning different behaviors to different roles — something that is impossible when you rely on a single prompt.

### How to run this cell

1. Run this cell after defining schemas and tools.
2. Read through each agent definition before executing.
3. Pay attention to how the `goal` and `backstory` shape the behavior of each agent.

### What should happen after running

The cell should create all three agent objects and print a confirmation message.

> **Key takeaway:** What makes a multi-agent system different from "one big prompt" is **role clarity** — each agent has a focused job, a distinct personality, and access only to the tools it needs.

In [ ]:
base_llm = LLM(model=MODEL_NAME, temperature=0.2)
creative_llm = LLM(model=MODEL_NAME, temperature=0.6)

researcher_goal = (
    "Collect five current, source-backed insights on the topic using the available search tool."
    if research_tools
    else "Collect five high-value insights on the topic and clearly flag any freshness limitations."
)

researcher = Agent(
    role="Research Analyst",
    goal=researcher_goal,
    backstory="You synthesize economic evidence clearly, avoid unsupported claims, and keep notes concise.",
    tools=research_tools,
    allow_delegation=False,
    llm=base_llm,
    verbose=True,
    inject_date=True,
    max_iter=8,
)

writer = Agent(
    role="Economic Content Writer",
    goal="Turn research notes into a crisp, accurate article that stays grounded in the provided evidence.",
    backstory="You write concise explainers for educated non-specialists and avoid padding.",
    allow_delegation=False,
    llm=creative_llm,
    verbose=True,
    max_iter=6,
)

social_agent = Agent(
    role="Social Media Editor",
    goal="Turn the final article into platform-appropriate draft posts without publishing anything.",
    backstory="You adapt long-form content into concise social copy while preserving the underlying claims.",
    allow_delegation=False,
    llm=creative_llm,
    verbose=True,
    max_iter=6,
)

print("Agents created successfully.")

Agents created successfully.


## Cell 7 — Define the task chain and handoffs

This cell defines the actual workflow that the agents will follow:

1. The **researcher** produces research notes.
2. The **writer** turns those notes into a structured article.
3. The **social editor** turns the article into platform-specific draft posts.

This is where the notebook moves from "three separate agents" to "a coordinated multi-agent pipeline."

### Why this matters

An agent system is usually built from two layers:

- the **agents** themselves (who can do the work),
- the **tasks** that route information between them (what work gets done, and in what order).

Tasks are the operational layer. They determine what each agent is asked to do, what output is expected, and what context should flow from one step into the next. Notice how later tasks reference earlier tasks through `context=[...]` — that is how CrewAI passes outputs forward through the pipeline.

This cell also demonstrates a useful real-world pattern: if live search is available, the research task uses it; if not, the instructions change so the researcher clearly states freshness limitations. That is an example of **graceful degradation** in agent design — the system adjusts to its environment instead of crashing.

### How to run this cell

1. Run this cell after the agents are created.
2. Review each task description carefully.
3. Notice how later tasks reference earlier tasks through `context=[...]`.

### What should happen after running

The cell will create three task objects (`research_task`, `article_task`, `social_task`). These tasks are now ready to be executed by a crew.

> **Key takeaway:** This notebook uses a **sequential process** where each step depends on prior context. CrewAI also supports parallel and hierarchical processes, but sequential is the clearest starting point for understanding multi-agent orchestration.

In [ ]:
if research_tools:
    research_description = dedent(f"""
    Research the topic "{TOPIC}" using the web search tool.
    Produce five concise insights that cover long-run growth, productivity, demographics, sector mix, and major risks/opportunities.
    Each insight must include a short source reference with the institution or publication name and year.
    Use only claims that you can support from the tool results.
    """).strip()
else:
    research_description = dedent(f"""
    Research the topic "{TOPIC}" using your existing model knowledge only.
    Produce five concise insights that cover long-run growth, productivity, demographics, sector mix, and major risks/opportunities.
    Do not invent URLs or precise citations. If freshness is uncertain, explicitly label that point as "freshness not verified".
    """).strip()

research_task = Task(
    description=research_description,
    expected_output="Five bullet points with concise explanations and clearly labeled source references or freshness caveats.",
    agent=researcher,
    markdown=True,
)

article_task = Task(
    description=dedent("""
    Turn the research notes into a structured article draft.

    Return:
    - title: a concise headline
    - summary: two or three sentences for busy readers
    - article_markdown: a 450-650 word markdown article body with an introduction, 3-4 subheadings, and a conclusion

    Keep every claim grounded in the research context.
    Do not invent facts that are not present upstream.
    Do not wrap the article in triple backticks.
    """).strip(),
    expected_output="A structured article draft with title, summary, and article_markdown fields.",
    agent=writer,
    context=[research_task],
    output_pydantic=ArticleDraft,
)

social_task = Task(
    description=dedent("""
    Use the article context to create draft social posts only.

    Return:
    - linkedin_post: 2-4 short paragraphs plus 3-5 relevant hashtags
    - x_post: one post no longer than 280 characters
    - publishing_status: exactly "Draft only — not published."

    Do not call any external publishing service.
    """).strip(),
    expected_output="A structured set of social post drafts and a draft-only status line.",
    agent=social_agent,
    context=[article_task],
    output_pydantic=SocialDrafts,
)

print("Tasks created successfully.")

Tasks created successfully.


## Cell 8 — Run the crew and inspect the outputs

This is the main execution cell of the notebook. It creates the crew, runs the full multi-agent workflow, collects the results, displays them in the notebook, and saves them to disk.

This is the point where all earlier design decisions come together: configuration, secret management, tools, agents, tasks, and structured outputs.

### What happens inside this cell

The `Crew` object is the **orchestration layer**. It decides which agents are included, which tasks are included, and how the process should run. This notebook uses `Process.sequential`, which means the tasks run in order and pass context forward.

After execution, the notebook extracts raw research notes, the structured article output, and the structured social output, then saves markdown files for later review.

### How to run this cell

1. Make sure all previous cells have already run successfully.
2. Run this cell and wait — **this is the longest-running step** in the notebook.
3. Do not interrupt it unless there is a clear error.
4. Once complete, scroll through the notebook output and review the research notes, article draft, social drafts, saved file paths, and token usage.

### What should happen after running

You should see:

- CrewAI execution logs (showing each agent's reasoning process),
- research notes rendered in markdown,
- an article rendered in markdown,
- social drafts rendered in markdown,
- saved output paths inside the `output/` folder,
- and token usage information.

### Troubleshooting

If this cell fails, check:

- whether the API key was loaded correctly (rerun Cell 3),
- whether the model name is valid for your account,
- whether a runtime restart was skipped after the install step,
- or whether rate limits or API access issues are preventing the model call.

> **Key takeaway:** The full lifecycle of a multi-agent system is **design → execute → inspect → save → review**. That workflow is often more important in practice than the prompt text alone.

In [ ]:
crew = Crew(
    name="economic-growth-demo",
    agents=[researcher, writer, social_agent],
    tasks=[research_task, article_task, social_task],
    process=Process.sequential,
    verbose=True,
    share_crew=False,
)

print("🚀 Starting CrewAI workflow...")
result = crew.kickoff()

if not result or not result.tasks_output:
    raise RuntimeError(
        "The crew returned no task outputs. Check the logs above for API errors, "
        "rate limits, or invalid model names."
    )

# Save the research notes.
research_notes_text = result.tasks_output[0].raw
research_notes_path = OUTPUT_DIR / "research_notes.md"
research_notes_path.write_text(research_notes_text, encoding="utf-8")

# Extract structured outputs when available.
article_structured = next(
    (task_output.pydantic for task_output in result.tasks_output if isinstance(task_output.pydantic, ArticleDraft)),
    None,
)
social_structured = next(
    (task_output.pydantic for task_output in result.tasks_output if isinstance(task_output.pydantic, SocialDrafts)),
    None,
)

if article_structured:
    article_text = (
        f"# {article_structured.title}\n\n"
        f"**Summary:** {article_structured.summary}\n\n"
        f"{article_structured.article_markdown}"
    )
else:
    article_text = result.tasks_output[1].raw

if social_structured:
    social_text = (
        "## LinkedIn\n\n"
        f"{social_structured.linkedin_post}\n\n"
        "## X / Twitter\n\n"
        f"{social_structured.x_post}\n\n"
        f"**Status:** {social_structured.publishing_status}"
    )
else:
    social_text = result.tasks_output[2].raw

article_path = OUTPUT_DIR / "article.md"
social_path = OUTPUT_DIR / "social_posts.md"

article_path.write_text(article_text, encoding="utf-8")
social_path.write_text(social_text, encoding="utf-8")

display(Markdown("## Research Notes"))
display(Markdown(research_notes_text))

display(Markdown("## Article"))
display(Markdown(article_text))

display(Markdown("## Social Drafts"))
display(Markdown(social_text))

print("\nSaved files:")
print(f"- Research notes: {research_notes_path.resolve()}")
print(f"- Article: {article_path.resolve()}")
print(f"- Social drafts: {social_path.resolve()}")

print("\nToken usage:")
print(result.token_usage)

🚀 Starting CrewAI workflow...


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.12.2                                                                                        │
│  Latest version:  1.14.6                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: economic-growth-demo                                                                                     │
│  ID: f07d2ab9-5179-4a9a-a28f-3fa18211b230                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research the topic "Agentic Analytics for Data Analysts" using your existing model knowledge only.       │
│  Produce five concise insights that cover long-run growth, productivity, demographics, sector mix, and major    │
│  risks/opportunities.                                                                                           │
│  Do not invent URLs or precise citations. If freshness is uncertain, explicitly label that point as "freshness  │
│  not verified".                                                                                                 │
│                                                                                                                 │
│  Current Date: 2026-06-11                                                                                       │
│  ID: 79c4d6dd-0a42-48bd-afa3-04290b88d26b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Analyst                                                                                        │
│                                                                                                                 │
│  Task: Research the topic "Agentic Analytics for Data Analysts" using your existing model knowledge only.       │
│  Produce five concise insights that cover long-run growth, productivity, demographics, sector mix, and major    │
│  risks/opportunities.                                                                                           │
│  Do not invent URLs or precise citations. If freshness is uncertain, explicitly label that point as "freshness  │
│  not verified".                                                                                                 │
│                                                                                                                 │
│  Current Date: 2026-06-11                                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

## Article

# The Rise of Agentic Analytics: Transforming the Data Landscape

**Summary:** Agentic analytics is poised to revolutionize data analytics by enhancing decision-making and productivity among analysts. As younger professionals enter the workforce and industries evolve, the adoption of these advanced tools presents both opportunities and risks that will shape the future of data-driven strategies.

# The Rise of Agentic Analytics: Transforming the Data Landscape

## Introduction
Agentic analytics represents a significant evolution in the field of data analytics, aiming to empower analysts to actively influence outcomes rather than merely interpret data. This innovative approach is expected to enhance decision-making capabilities and drive long-run growth in various sectors.

## Long-Run Growth
The integration of agentic analytics is anticipated to catalyze long-run growth in the data analytics domain. By equipping analysts with tools that enhance their decision-making capabilities, organizations can expect to see a shift towards more innovative solutions. This empowerment not only leads to improved business performance but also encourages a culture of proactive data utilization, where analysts can influence outcomes directly.

## Boosting Productivity
One of the most significant advantages of agentic analytics is its potential to boost productivity among data analysts. By automating routine tasks and providing sophisticated data interpretation tools, analysts can redirect their focus from mundane data wrangling to strategic initiatives. This shift is expected to increase overall efficiency, allowing organizations to leverage their data resources more effectively.

## Demographic Influences
The adoption of agentic analytics is likely to be shaped by demographic shifts in the workforce. As younger, tech-savvy professionals enter the job market, their inclination to adopt and utilize advanced analytics tools will drive demand for agentic analytics solutions. This generational transition is crucial, as it may lead to a more widespread acceptance of innovative technologies in data analytics, ultimately transforming industry practices.

## Sector Adoption Variability
Different sectors are poised to embrace agentic analytics at varying rates, predominantly influenced by the data-intensive nature of their operations. Industries such as finance, healthcare, and retail are expected to lead the charge in adopting these advanced analytics tools. The diverse sector mix will influence the development of tailored analytics solutions, ensuring that tools meet the specific needs and challenges of each industry.

## Risks and Opportunities
While the prospects of agentic analytics are promising, they are accompanied by notable risks. Data privacy concerns and the potential for over-reliance on automated systems pose significant challenges that organizations must navigate. However, the opportunities presented by enhanced predictive capabilities and the ability to derive actionable insights from complex datasets are substantial. Organizations that effectively leverage these advantages can gain a competitive edge in their respective markets.

## Conclusion
Agentic analytics is set to transform the landscape of data analytics by enhancing productivity and decision-making among analysts. As demographic trends and sector-specific needs shape the adoption of these advanced tools, organizations must remain vigilant about the associated risks. Embracing the opportunities presented by agentic analytics can lead to innovative solutions and improved performance in an increasingly data-driven world.

## Social Drafts

## LinkedIn

Agentic analytics is revolutionizing the data landscape by empowering analysts to actively influence outcomes rather than just interpreting data. This innovative approach enhances decision-making capabilities and fosters long-run growth in various sectors. 

As younger, tech-savvy professionals enter the workforce, the demand for advanced analytics tools is surging, driving the adoption of agentic analytics across industries like finance, healthcare, and retail. However, organizations must also navigate risks such as data privacy and over-reliance on automation. 

Embracing agentic analytics not only boosts productivity by automating routine tasks but also leads to innovative solutions that can give companies a competitive edge. The future of data-driven strategies is here! 

#DataAnalytics #AgenticAnalytics #Innovation #Productivity #DataDriven

## X / Twitter

Agentic analytics is transforming data analytics by empowering analysts to influence outcomes! 🚀 With the rise of tech-savvy professionals, industries are set to embrace these tools. But beware of risks like data privacy! #DataAnalytics #Innovation

**Status:** Draft only — not published.


Saved files:
- Research notes: /content/output/research_notes.md
- Article: /content/output/article.md
- Social drafts: /content/output/social_posts.md

Token usage:
total_tokens=6255 prompt_tokens=4261 cached_prompt_tokens=0 completion_tokens=1994 successful_requests=5


## Cell 9 — Optional manual publishing step

This final cell is optional and is included to demonstrate a safe deployment pattern.

The notebook does **not** publish automatically. Instead, it follows a **review-first workflow**:

1. Generate drafts.
2. Inspect the drafts.
3. Publish only if a human explicitly opts in.

This is a strong default for both classroom demonstrations and real-world AI workflows.

### Why this matters

In many agent systems, generation and action should be separated. Just because a model can generate content does not mean the system should immediately post it, send it, or trigger an external action. Adding a **manual approval step** is often the safer design choice — and it is the pattern you will encounter most often in production agent deployments.

### How to run this cell

1. By default, simply run the cell as is — it will confirm that publishing is disabled.
2. Only if you deliberately want to test publishing:
   - make sure `SOCIAL_WEBHOOK_URL` is loaded,
   - make sure Cell 8 has already run successfully,
   - set `MANUAL_PUBLISH = True`,
   - choose which draft to publish (`"linkedin"` or `"x"`),
   - then rerun the cell.

### What should happen after running

- If `MANUAL_PUBLISH` is `False`, the cell will print a safety message.
- If `MANUAL_PUBLISH` is `True`, the selected draft will be sent to the configured webhook.

> **Key takeaway:** When should an AI system only generate content? When should it be allowed to act? Where should human approval sit inside the workflow? These are design questions worth thinking about for every agent system you build.

In [ ]:
MANUAL_PUBLISH = False
POST_TO_PUBLISH = "linkedin"  # Choose "linkedin" or "x".

if MANUAL_PUBLISH:
    if not SOCIAL_WEBHOOK_URL:
        raise ValueError("SOCIAL_WEBHOOK_URL is not set.")

    if "social_structured" not in globals() or social_structured is None:
        raise ValueError("Run Cell 8 first so the draft posts exist.")

    post_text = (
        social_structured.linkedin_post
        if POST_TO_PUBLISH.lower() == "linkedin"
        else social_structured.x_post
    )

    print(publish_to_webhook(post_text))
else:
    print(
        "Manual publishing is disabled. Review output/social_posts.md first, then set "
        "MANUAL_PUBLISH = True if you really want to send one draft."
    )

Manual publishing is disabled. Review output/social_posts.md first, then set MANUAL_PUBLISH = True if you really want to send one draft.
